# Построение всех 6 LanceDB-баз (локально)

Один прогон → 6 таблиц в `lancedb_store/`: для трёх моделей
(rosberta, e5-base, bge-m3) × двух вариантов (base / fine-tuned).

**Ключевая гарантия:** во всех 6 таблицах ровно одни и те же 50 000 постов
(одни и те же `post_id`, в одном и том же порядке). Семплинг идёт один раз,
до загрузки моделей, с фиксированным `RANDOM_SEED = 42`. После этого тот же
список `posts` переиспользуется для каждой модели — меняется только вектор.

## Перед запуском

1. Убедись, что в секции `MODELS` ниже все пути ведут к моделям, которые
   у тебя есть локально. Закомментируй или поставь `enabled=False` для тех,
   которых ещё нет.
2. Скачать с HF (`intfloat/multilingual-e5-base`, `deepvk/USER-bge-m3`,
   `ai-forever/ru-en-RoSBERTa`) ноутбук попытается сам, если есть интернет.
3. Уже существующие таблицы с тем же именем будут пересоздаваться (`drop`).
   Поэтому если хочешь перестроить только одну — отключи остальные через
   `enabled=False`.


In [1]:
# Локальный ноутбук перенесён в подпапку — восстанавливаем CWD в корень thesis/
import os
from pathlib import Path

if Path.cwd().name != "thesis":
    os.chdir("..")
print("CWD:", Path.cwd())


CWD: c:\Users\Admin\Documents\диплом\thesis


In [2]:
# Окружение
import warnings
warnings.filterwarnings('ignore')

import os
import torch

# Совместимость sentence-transformers 5.x + transformers 4.57
import transformers
from transformers.modeling_utils import PreTrainedModel
transformers.PreTrainedModel = PreTrainedModel

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    # CPU-оптимизация: используем число ФИЗИЧЕСКИХ ядер, не логических
    # (hyperthreading на BERT-инференсе даёт отрицательный эффект)
    try:
        import psutil
        n_phys = psutil.cpu_count(logical=False) or (os.cpu_count() // 2)
    except ImportError:
        n_phys = max(1, (os.cpu_count() or 2) // 2)
    torch.set_num_threads(n_phys)
    torch.set_num_interop_threads(1)
    # MKL/OpenBLAS тоже прижимаем — чтобы торч не конкурировал сам с собой
    os.environ['OMP_NUM_THREADS']  = str(n_phys)
    os.environ['MKL_NUM_THREADS']  = str(n_phys)
    print(f'GPU не найден — CPU-инференс, {n_phys} потоков.')


GPU не найден — CPU-инференс, 8 потоков.


In [3]:
# ==================== MODELS: 6 пресетов ====================
# Каждая запись — отдельная LanceDB-таблица. Можно отключить любую через enabled=False.
# `source` бывает трёх видов:
#   1) HF-идентификатор ("intfloat/multilingual-e5-base") — скачается в кэш
#   2) локальный путь к папке с моделью (например, models/final/bi-encoder)
#   3) локальный путь к .tar.gz (распакуется в models/_extracted/)

MODELS = [
    {
        "name":         "rosberta-base",
        "table_name":   "rosberta-base-50k",
        "source":       "ai-forever/ru-en-RoSBERTa",
        "doc_prefix":   "",
        "query_prefix": "",
        "batch_size":   512,
        "enabled":      True,
    },
    {
        "name":         "rosberta-fine-tuned",
        "table_name":   "rosberta-fine-tuned-50k",
        "source":       "models/final/bi-encoder",
        "doc_prefix":   "",
        "query_prefix": "",
        "batch_size":   512,
        "enabled":      True,
    },
    {
        "name":         "e5-base-base",
        "table_name":   "e5-base-base-50k",
        "source":       "intfloat/multilingual-e5-base",
        "doc_prefix":   "passage: ",
        "query_prefix": "query: ",
        "batch_size":   256,
        "enabled":      True,
    },
    {
        "name":         "e5-base-fine-tuned",
        "table_name":   "e5-base-fine-tuned-50k",
        # Подставь путь к локально распакованной модели или .tar.gz
        "source":       "models/bi-encoder-e5-finetuned",
        "doc_prefix":   "passage: ",
        "query_prefix": "query: ",
        "batch_size":   256,
        "enabled":      True,
    },
    {
        "name":         "bge-m3-base",
        "table_name":   "bge-m3-base-50k",
        "source":       "deepvk/USER-bge-m3",
        "doc_prefix":   "",
        "query_prefix": "",
        "batch_size":   64,
        "enabled":      True,
    },
    {
        "name":         "bge-m3-fine-tuned",
        "table_name":   "bge-m3-fine-tuned-50k",
        "source":       "models/bi-encoder-bge-m3-finetuned",
        "doc_prefix":   "",
        "query_prefix": "",
        "batch_size":   64,
        "enabled":      True,
    },
]

# --------- общие параметры (одинаковые для всех моделей) ---------
MAX_POSTS      = 10000
# ВАЖНО: max_seq_length НЕ задаём — используем архитектурный максимум модели
# (обычно 512 токенов). Посты длиннее этого обрезает сам трансформер.
MAX_SEQ_LEN    = None
RANDOM_SEED    = 42
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"
CLEAN_CATEGORY = True      # оставить только первую категорию до '|||'

enabled_count = sum(1 for m in MODELS if m['enabled'])
print(f"Включено моделей: {enabled_count} из {len(MODELS)}")
print(f"DEVICE: {DEVICE}")
for m in MODELS:
    flag = '✓' if m['enabled'] else '·'
    print(f"  {flag} {m['name']:25s} → {m['table_name']}")


Включено моделей: 6 из 6
DEVICE: cpu
  ✓ rosberta-base             → rosberta-base-50k
  ✓ rosberta-fine-tuned       → rosberta-fine-tuned-50k
  ✓ e5-base-base              → e5-base-base-50k
  ✓ e5-base-fine-tuned        → e5-base-fine-tuned-50k
  ✓ bge-m3-base               → bge-m3-base-50k
  ✓ bge-m3-fine-tuned         → bge-m3-fine-tuned-50k


In [4]:
# Локальные пути
POSTS_FILE       = "data/posts/ajtkulov/selected/selected500k_cleaned.jsonl"
LANCEDB_PATH     = "./lancedb_store"
EXTRACTED_MODELS = "models/_extracted"   # сюда распаковываются .tar.gz

os.makedirs(LANCEDB_PATH, exist_ok=True)
os.makedirs(EXTRACTED_MODELS, exist_ok=True)

assert os.path.exists(POSTS_FILE), f"Входной файл не найден: {POSTS_FILE}"
print(f"Входной файл: {POSTS_FILE}  ({os.path.getsize(POSTS_FILE)/1e6:.1f} MB)")
print(f"LanceDB: {LANCEDB_PATH}")


Входной файл: data/posts/ajtkulov/selected/selected500k_cleaned.jsonl  (378.5 MB)
LanceDB: ./lancedb_store


In [5]:
# ============================================================
# СЕМПЛИНГ ПОСТОВ — выполняется ОДИН РАЗ, до загрузки моделей.
# Эти 50 000 постов будут использованы для всех 6 таблиц.
# ============================================================
import json, random
from tqdm.auto import tqdm

all_posts = []
with open(POSTS_FILE, 'r', encoding='utf-8') as f:
    for line in tqdm(f, desc="Чтение постов"):
        obj = json.loads(line)
        if not obj.get('text', '').strip():
            continue
        if CLEAN_CATEGORY:
            cat = obj.get('category', '')
            if '|||' in cat:
                obj['category'] = cat.split('|||')[0].strip()
        all_posts.append(obj)

print(f"Всего постов в файле: {len(all_posts):,}")

rng = random.Random(RANDOM_SEED)
posts = rng.sample(all_posts, MAX_POSTS)
del all_posts   # больше не нужны — освобождаем память

def make_post_id(post):
    return f"{post['channel']}::{post['id']}"

post_ids = [make_post_id(p) for p in posts]
print(f"Случайная выборка (seed={RANDOM_SEED}): {len(posts):,}")
print(f"Первый post_id:  {post_ids[0]}")
print(f"Последний post_id: {post_ids[-1]}")
print(f"Уникальных post_id: {len(set(post_ids)):,}  (должно быть = {MAX_POSTS:,})")

from collections import Counter
channels = set(p['channel'] for p in posts)
categories = Counter(p['category'] for p in posts)
print(f"Уникальных каналов:    {len(channels):,}")
print(f"Уникальных категорий:  {len(categories)}")


Чтение постов: 0it [00:00, ?it/s]

Всего постов в файле: 499,855
Случайная выборка (seed=42): 10,000
Первый post_id:  obshakstaya::767
Последний post_id: fashion_mur::9053
Уникальных post_id: 9,142  (должно быть = 10,000)
Уникальных каналов:    6,427
Уникальных категорий:  39


In [6]:
# Универсальный распаковщик/резолвер моделей
import tarfile, time

def resolve_model(source: str) -> str:
    """Превращает source из MODELS в путь, который понимает SentenceTransformer.

    Порядок проверок:
      1) если source — существующий локальный архив .tar.gz → распаковать и вернуть папку
      2) если source — существующая локальная папка → вернуть как есть
      3) иначе если выглядит как HF-id (org/name, без точек/слэшей в начале)
         → отдать как есть, SentenceTransformer сам скачает с HF
      4) иначе — assert: ничего не найдено
    """
    # 1) Архив .tar.gz
    if source.endswith(".tar.gz") and os.path.exists(source):
        base_name = os.path.basename(source)[:-len(".tar.gz")]
        extracted = os.path.join(EXTRACTED_MODELS, base_name)
        if os.path.exists(extracted) and os.listdir(extracted):
            print(f"  модель уже распакована: {extracted}")
            return extracted
        print(f"  распаковка {source} → {extracted} ...")
        t0 = time.time()
        with tarfile.open(source, "r:gz") as tar:
            tar.extractall(EXTRACTED_MODELS)
        print(f"    готово за {time.time()-t0:.0f}с")
        assert os.path.isdir(extracted), (
            f"Ожидалась папка {extracted} после распаковки, но её нет. "
            f"Содержимое {EXTRACTED_MODELS}: {os.listdir(EXTRACTED_MODELS)}"
        )
        return extracted

    # 2) Существующая локальная папка (относительный или абсолютный путь)
    if os.path.isdir(source):
        return source

    # 3) HF-идентификатор: org/name, без расширения .tar.gz, без явного локального префикса
    looks_like_hf = (
        "/" in source
        and not source.startswith(".")
        and not source.startswith("/")
        and not source.endswith(".tar.gz")
    )
    if looks_like_hf:
        return source

    # 4) Ничего не подошло
    raise FileNotFoundError(
        f"Не удалось разрешить source: {source!r}. "
        f"Это не существующий .tar.gz, не существующая папка и не похоже на HF-id."
    )


In [7]:
# Функция, которая строит ОДНУ таблицу из общего списка `posts`
import gc, time
import pyarrow as pa
import lancedb
from sentence_transformers import SentenceTransformer

def build_table_for_model(cfg, posts, lancedb_path):
    """Загружает модель cfg, кодирует posts, пишет в таблицу cfg['table_name']."""
    name        = cfg['name']
    table_name  = cfg['table_name']
    source      = cfg['source']
    doc_prefix  = cfg['doc_prefix']
    batch_size  = cfg['batch_size']

    print(f"\n{'=' * 70}")
    print(f"  МОДЕЛЬ: {name}  →  таблица {table_name}")
    print(f"{'=' * 70}")

    model_path = resolve_model(source)
    print(f"  source: {source}")
    print(f"  path:   {model_path}")

    bi_encoder = SentenceTransformer(model_path, device=DEVICE)
    # max_seq_length выставляем ТОЛЬКО если MAX_SEQ_LEN не None —
    # иначе оставляем архитектурный дефолт модели (обычно 512)
    if MAX_SEQ_LEN is not None:
        bi_encoder.max_seq_length = MAX_SEQ_LEN
    dim = bi_encoder.get_sentence_embedding_dimension()
    print(f"  dim={dim}, max_seq_len={bi_encoder.max_seq_length}, batch_size={batch_size}")

    db = lancedb.connect(lancedb_path)

    schema = pa.schema([
        pa.field("vector",   pa.list_(pa.float32(), dim)),
        pa.field("text",     pa.utf8()),
        pa.field("channel",  pa.utf8()),
        pa.field("category", pa.utf8()),
        pa.field("post_id",  pa.utf8()),
        pa.field("link",     pa.utf8()),
        pa.field("date",     pa.utf8()),
        pa.field("views",    pa.utf8()),
    ])

    if table_name in db.table_names():
        db.drop_table(table_name)
        print(f"  старая таблица '{table_name}' удалена")

    table = db.create_table(table_name, schema=schema)

    total = len(posts)
    t_start = time.time()
    indexed = 0
    pbar = tqdm(total=total, desc=f"  {name}", unit="пост")

    for batch_start in range(0, total, batch_size):
        batch = posts[batch_start : batch_start + batch_size]
        texts = [p['text'] for p in batch]
        encoded_texts = [doc_prefix + t for t in texts] if doc_prefix else texts

        embeddings = bi_encoder.encode(
            encoded_texts,
            normalize_embeddings=True,
            show_progress_bar=False,
            batch_size=batch_size,
            device=DEVICE,
        )

        records = [{
            "vector":   embeddings[i].tolist(),
            "text":     batch[i]['text'],
            "channel":  batch[i]['channel'],
            "category": batch[i].get('category', ''),
            "post_id":  make_post_id(batch[i]),
            "link":     batch[i].get('link', ''),
            "date":     batch[i].get('date', ''),
            "views":    str(batch[i].get('views', '')),
        } for i in range(len(batch))]

        table.add(records)
        indexed += len(records)
        elapsed = time.time() - t_start
        speed = indexed / elapsed if elapsed > 0 else 0
        eta = (total - indexed) / speed if speed > 0 else 0
        pbar.update(len(records))
        pbar.set_postfix({"скор": f"{speed:.0f} п/с", "ETA": f"{eta/60:.1f}м"})
    pbar.close()

    print(f"  векторизация: {indexed:,} постов за {time.time()-t_start:.0f}с")
    print(f"  создание FTS-индекса...")
    table.create_fts_index("text", replace=True)
    print(f"  ✓ таблица {table_name}: {table.count_rows():,} строк, dim={dim}")

    # Освобождаем VRAM перед следующей моделью
    del bi_encoder
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "name":       name,
        "table_name": table_name,
        "rows":       table.count_rows(),
        "dim":        dim,
        "seconds":    round(time.time() - t_start, 1),
    }


In [8]:
# ============================================================
# ГЛАВНЫЙ ЦИКЛ — строим таблицу за таблицей
# ============================================================
results = []
for cfg in MODELS:
    if not cfg['enabled']:
        print(f"\n[пропуск] {cfg['name']} — enabled=False")
        continue
    try:
        info = build_table_for_model(cfg, posts, LANCEDB_PATH)
        results.append(info)
    except Exception as e:
        print(f"  ✗ ОШИБКА для {cfg['name']}: {type(e).__name__}: {e}")
        results.append({"name": cfg['name'], "error": str(e)})

print("\n\n" + "=" * 70)
print("  СВОДКА")
print("=" * 70)
for r in results:
    if 'error' in r:
        print(f"  ✗ {r['name']}: {r['error']}")
    else:
        print(f"  ✓ {r['name']:25s} dim={r['dim']:5d}  rows={r['rows']:,}  {r['seconds']}с")



  МОДЕЛЬ: rosberta-base  →  таблица rosberta-base-50k
  source: ai-forever/ru-en-RoSBERTa
  path:   ai-forever/ru-en-RoSBERTa


Some weights of RobertaModel were not initialized from the model checkpoint at ai-forever/ru-en-RoSBERTa and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  dim=1024, max_seq_len=512, batch_size=512
  старая таблица 'rosberta-base-50k' удалена


  rosberta-base:   0%|          | 0/10000 [00:00<?, ?пост/s]

KeyboardInterrupt: 

In [ ]:
# ============================================================
# ВЕРИФИКАЦИЯ: во всех успешно построенных таблицах post_id-set одинаков
# ============================================================
db = lancedb.connect(LANCEDB_PATH)
expected_ids = set(post_ids)
print(f"Эталонный набор post_id: {len(expected_ids):,}")

for r in results:
    if 'error' in r:
        continue
    t = db.open_table(r['table_name'])
    # to_lance() — нижележащий Lance-датасет, у которого проекция по столбцам
    # есть с самых ранних версий (в отличие от LanceTable.to_pandas(columns=...))
    table_ids = set(t.to_lance().to_table(columns=['post_id']).to_pandas()['post_id'].tolist())
    n_match = len(table_ids & expected_ids)
    n_extra = len(table_ids - expected_ids)
    n_miss  = len(expected_ids - table_ids)
    status  = '✓' if (n_match == len(expected_ids) and n_extra == 0) else '✗'
    print(f"  {status} {r['table_name']:30s} match={n_match:,}  extra={n_extra}  missing={n_miss}")


In [ ]:
# Размеры таблиц на диске
def dir_size_mb(path):
    total = 0
    for dp, _, fs in os.walk(path):
        for f in fs:
            total += os.path.getsize(os.path.join(dp, f))
    return total / (1024 * 1024)

for r in results:
    if 'error' in r:
        continue
    table_dir = os.path.join(LANCEDB_PATH, f"{r['table_name']}.lance")
    size = dir_size_mb(table_dir)
    print(f"  {r['table_name']:30s} {size:7.1f} MB  ({size*1024/r['rows']:.1f} KB/запись)")


In [ ]:
# Санити-проверка: пробуем один и тот же запрос на каждой таблице
test_query = "Кроссовки Nike Air Max мужские для бега, размер 42, чёрные"
print(f"Запрос: {test_query}\n")

for cfg in MODELS:
    if not cfg['enabled']:
        continue
    if not any(r.get('table_name') == cfg['table_name'] and 'error' not in r for r in results):
        continue
    print(f"--- {cfg['name']} ---")
    t = db.open_table(cfg['table_name'])
    # Используем BM25 — не нужно загружать модель ещё раз
    rows = (t.search(test_query, query_type='fts')
            .limit(3).select(['text','channel','category']).to_list())
    for i, r in enumerate(rows, 1):
        print(f"  {i}. [{r['category']}] @{r['channel']}")
        print(f"     {r['text'][:120]}...")
    print()


## После прогона

В `thesis/lancedb_store/` лежат 6 папок `<table_name>.lance/`. Все 6 содержат
**один и тот же набор** `post_id` (проверка в верификационной ячейке выше).

Бенчмарки и тестовые ноутбуки теперь могут честно сравнивать модели —
разница в качестве поиска объясняется только эмбеддингами, не выборкой данных.

## Если что-то упало

В цикле построения каждая модель обёрнута в `try/except` — если одна модель
сломалась (например, не нашёлся файл), остальные продолжат строиться. После
разбора причины можно выставить `enabled=True` только для проблемной модели
и перезапустить ноутбук — остальные таблицы трогать не нужно.
